In [1]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from src.data import load_adult
from src.models import build_models
from src.explain import load_tuned_params
from src.fairness import group_report, gaps, attribute_predictability

os.makedirs("../results", exist_ok=True)
X_train, X_test, y_train, y_test = load_adult()
params = load_tuned_params()
cv = StratifiedKFold(5, shuffle=True, random_state=42)
sex = X_train["sex"].to_numpy()
y = y_train.to_numpy()

In [2]:
# Compute out-of-fold predictions for different feature variants
VARIANTS = {
    "all features": [],
    "drop sex, race": ["sex", "race"],
    "drop sex, race, relationship": ["sex", "race", "relationship"],
    "drop sex, race, relationship, marital-status":
        ["sex", "race", "relationship", "marital-status"],
}
oof = {}
for name, drop in VARIANTS.items():
    Xv = X_train.drop(columns=drop)
    model = build_models(Xv, xgb_params=params)["XGBoost (boosting)"]
    oof[name] = cross_val_predict(model, Xv, y_train, cv=cv,
                                  method="predict_proba", n_jobs=1)[:, 1]
    assert len(oof[name]) == len(X_train)
    print("done:", name, "| OOF AUC:", round(roc_auc_score(y, oof[name]), 4))
pd.DataFrame(oof).to_csv("../results/fairness_oof_predictions.csv", index=False)

ref = roc_auc_score(y, oof["all features"])
assert abs(ref - 0.9293) < 0.01, f"OOF AUC {ref:.4f} is far from the tuned CV AUC: check alignment"

done: all features | OOF AUC: 0.9289
done: drop sex, race | OOF AUC: 0.9282
done: drop sex, race, relationship | OOF AUC: 0.9271
done: drop sex, race, relationship, marital-status | OOF AUC: 0.8877


In [3]:
# Compute fairness metrics for each feature variant
point = {n: gaps(group_report(y, p, sex), "Male", "Female") for n, p in oof.items()}
print("base rate gap (M-F), a property of the labels:",
      round(point["all features"]["base_rate gap"], 4))
summary = pd.DataFrame({
    n: {"ROC-AUC": roc_auc_score(y, p),
        "F1@0.5": f1_score(y, (p >= 0.5).astype(int)),
        "selection gap": point[n]["selection_rate gap"],
        "TPR gap": point[n]["TPR gap"],
        "FPR gap": point[n]["FPR gap"]}
    for n, p in oof.items()}).T
summary.round(4)

base rate gap (M-F), a property of the labels: 0.1949


,ROC-AUC,F1@0.5,selection gap,TPR gap,FPR gap
all features,0.9289,0.7090,0.1780,0.0801,0.0631
"drop sex, race",0.9282,0.7078,0.1743,0.0640,0.0616
"drop sex, race, relationship",0.9271,0.7072,0.1912,0.1377,0.0701
"drop sex, race, relationship, marital-status",0.8877,0.6169,0.1086,0.0699,0.0105


In [4]:
# Compute bootstrap confidence intervals for each gap
rng = np.random.default_rng(0)
B = 1000
idx = rng.integers(0, len(y), size=(B, len(y)))
keys = ["selection_rate gap", "TPR gap", "FPR gap"]
boot = {n: {k: np.empty(B) for k in keys} for n in oof}
for b in range(B):
    ii = idx[b]
    for n, p in oof.items():
        g = gaps(group_report(y[ii], p[ii], sex[ii]), "Male", "Female")
        for k in keys:
            boot[n][k][b] = g[k]
# a function to compute a 95% bootstrap confidence interval for a given array
def ci(a):
    return f"[{np.percentile(a, 2.5):+.3f}, {np.percentile(a, 97.5):+.3f}]"

base = "all features"
gap_table = pd.DataFrame(
    {n: {k: f"{point[n][k]:+.3f} {ci(boot[n][k])}" for k in keys} for n in oof}).T
delta_table = pd.DataFrame(
    {n: {k: f"{point[n][k] - point[base][k]:+.3f} {ci(boot[n][k] - boot[base][k])}"
         for k in keys} for n in oof if n != base}).T

summary.to_csv("../results/fairness_summary.csv")
gap_table.to_csv("../results/fairness_gaps_ci.csv")
delta_table.to_csv("../results/fairness_gap_changes_ci.csv")
print("gaps (male minus female), 95% bootstrap CI"); display(gap_table)
print("change in each gap vs all features (paired)"); display(delta_table)

gaps (male minus female), 95% bootstrap CI


,selection_rate gap,TPR gap,FPR gap
all features,"+0.178 [+0.171, +0.185]","+0.080 [+0.053, +0.107]","+0.063 [+0.059, +0.068]"
"drop sex, race","+0.174 [+0.168, +0.181]","+0.064 [+0.036, +0.091]","+0.062 [+0.057, +0.066]"
"drop sex, race, relationship","+0.191 [+0.185, +0.198]","+0.138 [+0.110, +0.165]","+0.070 [+0.066, +0.075]"
"drop sex, race, relationship, marital-status","+0.109 [+0.102, +0.115]","+0.070 [+0.042, +0.098]","+0.010 [+0.006, +0.015]"


change in each gap vs all features (paired)


,selection_rate gap,TPR gap,FPR gap
"drop sex, race","-0.004 [-0.006, -0.002]","-0.016 [-0.026, -0.007]","-0.001 [-0.003, +0.000]"
"drop sex, race, relationship","+0.013 [+0.011, +0.016]","+0.058 [+0.042, +0.074]","+0.007 [+0.005, +0.010]"
"drop sex, race, relationship, marital-status","-0.069 [-0.075, -0.063]","-0.010 [-0.043, +0.018]","-0.053 [-0.058, -0.047]"


In [5]:
# Check whether the sensitive attribute is predictable from the remaining features
leak = []
for name, drop in VARIANTS.items():
    a = attribute_predictability(X_train, "sex", "Male", drop)
    leak.append({"variant": name, "AUC (sex from remaining features)": a.mean(), "std": a.std()})
leak = pd.DataFrame(leak).set_index("variant")
leak.to_csv("../results/fairness_sex_leakage.csv")
leak.round(3)

,AUC (sex from remaining features),std
variant,,
all features,0.929,0.003
"drop sex, race",0.929,0.003
"drop sex, race, relationship",0.857,0.006
"drop sex, race, relationship, marital-status",0.791,0.005


In [6]:
# Compute per-group rates for each feature variant and save to CSV
sex_reports = []
for n, p in oof.items():
    r = group_report(y, p, sex)
    r.insert(0, "variant", n)
    sex_reports.append(r)
sex_reports = pd.concat(sex_reports)
sex_reports.to_csv("../results/fairness_by_sex.csv")

race = X_train["race"].to_numpy()
race_rep = group_report(y, oof["all features"], race)
race_rep["n_pos"] = (race_rep["base_rate"] * race_rep["n"]).round().astype(int)
race_rep.to_csv("../results/fairness_by_race_all_features.csv")

print(sex_reports[sex_reports["variant"].isin(["all features", "drop sex, race"])].round(3))
print(race_rep.round(3))

               variant      n  base_rate  selection_rate    TPR    FPR    AUC
Male      all features  26140      0.304           0.259  0.663  0.082  0.910
Female    all features  12933      0.109           0.081  0.582  0.019  0.949
Male    drop sex, race  26140      0.304           0.257  0.658  0.082  0.910
Female  drop sex, race  12933      0.109           0.083  0.594  0.020  0.948
                        n  base_rate  selection_rate    TPR    FPR    AUC  \
White               33450      0.254           0.214  0.656  0.063  0.925   
Black                3717      0.118           0.083  0.549  0.021  0.953   
Amer-Indian-Eskimo    374      0.112           0.067  0.452  0.018  0.908   
Other                 324      0.123           0.083  0.550  0.018  0.962   
Asian-Pac-Islander   1208      0.271           0.246  0.679  0.085  0.913   

                    n_pos  
White                8503  
Black                 437  
Amer-Indian-Eskimo     42  
Other                  40  
Asian-P